# Multi-resolution Event V2 — Relation V2 on one P100

Open the published private Notebook `vanila111/trauma-predict-relation-v2-p100-r9`, confirm that its frozen private Dataset is listed under Input, select **P100**, keep Internet off, and choose **Save & Run All**. A raw `.ipynb` import must have the same Dataset added before any non-interactive run. The Notebook verifies and installs the hash-locked PyTorch 2.10.0 CUDA 12.6 runtime before launching the unchanged six-M4 Relation V2 training contract.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys

DATASET_REF = 'vanila111/trauma-predict-relation-v2-p100-r9-bundle'
SCHEMA = 'trauma_predict.multires_event_v2_relation_v2_p100_bundle.v2'
RUNTIME_SCHEMA = 'trauma_predict.p100_torch_runtime_wheelhouse.v1'
RUNTIME_CONTRACT_SHA256 = 'aada1dee4ee21e02fd5c81ae97d441c38e72d770eec5398932ee295d08f8f2cc'
RUNTIME_INVENTORY_SHA256 = '8063e83b243589e26c353d335fd5137505bfa90b2d5aa0b1226c15fd810120a1'
RUNTIME_TORCH_VERSION = '2.10.0+cu126'
RUNTIME_CUDA_VERSION = '12.6'
RUNTIME_CUDA_ARCH = 'sm_60'
RUNTIME_ROOT = Path('/kaggle/working/relation_v2_p100_runtime_py312_cu126')
RUNTIME_MARKER = 'RUNTIME_READY.json'

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def sha256_payload(value):
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(',', ':')).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()

def find_bundles(root):
    matches = []
    if not root.is_dir():
        return matches
    for manifest_path in sorted(root.rglob('run_bundle_manifest.json')):
        if manifest_path.is_symlink() or not manifest_path.is_file():
            continue
        payload = json.loads(manifest_path.read_text(encoding='utf-8'))
        if payload.get('schema') == SCHEMA and payload.get('dataset_ref') == DATASET_REF:
            matches.append((manifest_path.parent.resolve(), payload))
    return matches

def resolve_file(bundle, row, label):
    relative = Path(str(row.get('path') or ''))
    if not relative.parts or relative.is_absolute() or '..' in relative.parts:
        raise ValueError(f'{label} has an invalid relative path')
    path = bundle / relative
    if path.is_symlink() or not path.is_file():
        raise FileNotFoundError(f'{label} is absent: {path}')
    if path.stat().st_size != int(row.get('size_bytes', -1)):
        raise ValueError(f'{label} size differs from the frozen lock')
    if sha256_file(path) != row.get('sha256'):
        raise ValueError(f'{label} hash differs from the frozen lock')
    return path

attached = find_bundles(Path('/kaggle/input'))
if len(attached) != 1:
    raise RuntimeError(
        f'Expected one pre-bound private Dataset under /kaggle/input, found {len(attached)}. '
        f'Add Input {DATASET_REF} before Save & Run; a non-interactive run cannot attach it dynamically.'
    )
bundle, manifest = attached[0]
print('RELATION_V2_P100_BOUND_INPUT_OK', bundle, flush=True)

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f'Frozen P100 runtime requires Python 3.12, found {sys.version}')
runtime = manifest.get('runtime')
if not isinstance(runtime, dict):
    raise TypeError('Bundle manifest lacks its frozen runtime object')
contract_row = runtime.get('contract')
if not isinstance(contract_row, dict) or contract_row.get('sha256') != RUNTIME_CONTRACT_SHA256:
    raise ValueError('Bundle runtime contract identity differs from the frozen Notebook')
contract_path = resolve_file(bundle, contract_row, 'P100 runtime contract')
contract = json.loads(contract_path.read_text(encoding='utf-8'))
expected_scalars = {
    'schema': RUNTIME_SCHEMA,
    'python_abi': 'cp312',
    'torch_version': RUNTIME_TORCH_VERSION,
    'cuda_version': RUNTIME_CUDA_VERSION,
    'required_cuda_arch': RUNTIME_CUDA_ARCH,
    'inventory_sha256': RUNTIME_INVENTORY_SHA256,
}
for key, value in expected_scalars.items():
    if runtime.get(key) != value or contract.get(key) != value:
        raise ValueError(f'Frozen runtime scalar differs for {key}')
rows = runtime.get('files')
if not isinstance(rows, list) or rows != contract.get('files') or len(rows) != 28:
    raise ValueError('Runtime wheel rows differ from the frozen contract')
inventory = {
    'schema': RUNTIME_SCHEMA,
    'python_abi': 'cp312',
    'torch_version': RUNTIME_TORCH_VERSION,
    'cuda_version': RUNTIME_CUDA_VERSION,
    'required_cuda_arch': RUNTIME_CUDA_ARCH,
    'files': rows,
}
if sha256_payload(inventory) != RUNTIME_INVENTORY_SHA256:
    raise ValueError('Runtime wheel inventory digest differs from the frozen contract')
wheel_paths = [resolve_file(bundle, row, f'runtime wheel {index}') for index, row in enumerate(rows)]
print(f'RELATION_V2_P100_RUNTIME_WHEELS_OK files={len(wheel_paths)}', flush=True)

ready_payload = {
    'runtime_contract_sha256': RUNTIME_CONTRACT_SHA256,
    'runtime_inventory_sha256': RUNTIME_INVENTORY_SHA256,
    'torch_version': RUNTIME_TORCH_VERSION,
}
if RUNTIME_ROOT.exists():
    marker = RUNTIME_ROOT / RUNTIME_MARKER
    if marker.is_symlink() or not marker.is_file() or json.loads(marker.read_text(encoding='utf-8')) != ready_payload:
        raise RuntimeError('Existing isolated P100 runtime is not the frozen cu126 installation')
else:
    temporary = RUNTIME_ROOT.with_name(f'.{RUNTIME_ROOT.name}.install-{os.getpid()}')
    if temporary.exists():
        shutil.rmtree(temporary)
    command = [
        sys.executable, '-m', 'pip', 'install',
        '--no-index', '--no-deps', '--no-input', '--no-compile', '--no-cache-dir',
        '--disable-pip-version-check', '--target', str(temporary),
        *[str(path) for path in wheel_paths],
    ]
    subprocess.run(command, check=True)
    (temporary / RUNTIME_MARKER).write_text(
        json.dumps(ready_payload, indent=2, sort_keys=True) + '\n', encoding='utf-8'
    )
    temporary.replace(RUNTIME_ROOT)
print('RELATION_V2_P100_RUNTIME_INSTALLED', RUNTIME_ROOT, flush=True)

runtime_env = os.environ.copy()
inherited_pythonpath = runtime_env.get('PYTHONPATH', '').strip()
runtime_env['PYTHONPATH'] = os.pathsep.join(
    part for part in (str(RUNTIME_ROOT), inherited_pythonpath) if part
)
runtime_env['PYTHONNOUSERSITE'] = '1'
runtime_env['TRAUMA_PREDICT_RUNTIME_SITE_PACKAGES'] = str(RUNTIME_ROOT)
runtime_env['TRAUMA_PREDICT_RUNTIME_LOCK_SHA256'] = RUNTIME_CONTRACT_SHA256
runtime_env['PIP_NO_INDEX'] = '1'

smoke = '''
import torch
assert str(torch.__version__) == '2.10.0+cu126', torch.__version__
assert str(torch.version.cuda) == '12.6', torch.version.cuda
assert torch.cuda.is_available() and torch.cuda.device_count() == 1
assert 'P100' in torch.cuda.get_device_name(0).upper(), torch.cuda.get_device_name(0)
assert tuple(torch.cuda.get_device_capability(0)) == (6, 0)
assert 'sm_60' in torch.cuda.get_arch_list(), torch.cuda.get_arch_list()
layer = torch.nn.Linear(8, 8, device='cuda', dtype=torch.float16)
value = torch.arange(32, device='cuda', dtype=torch.float16).reshape(4, 8) / 32
loss = layer(value).float().square().mean()
loss.backward()
torch.cuda.synchronize()
assert all(parameter.grad is not None and bool(torch.isfinite(parameter.grad).all().item()) for parameter in layer.parameters())
print('RELATION_V2_P100_CU126_CUDA_BACKWARD_OK', torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0), flush=True)
'''
subprocess.run([sys.executable, '-c', smoke], env=runtime_env, check=True)
launcher = bundle / 'run_relation_v2_p100_bundle.py'
if launcher.is_symlink() or not launcher.is_file():
    raise FileNotFoundError(f'Bundle lacks its manifest-bound launcher: {launcher}')
subprocess.run(
    [sys.executable, str(launcher), '--bundle-root', str(bundle)],
    env=runtime_env,
    check=True,
)
